In [12]:
import ee
import geemap

In [13]:
ee.Authenticate()
ee.Initialize(project='ee-ojasvibansal')

In [14]:
CROP_CLASSES = [10,         # Rainfed cropland
                11,         # Herbaceous cover cropland
                12,         # Orchard cropland
                20          # Irrigated cropland
]

TREE_CLASSES = [
    51, 52,                 # Evergreen broadleaved forest
    61, 62,                 # Deciduous broadleaved forest
    71, 72,                 # Evergreen needle-leaved forest
    81, 82,                 # Deciduous needle-leaved forest
    91, 92                  # Mixed forest
]

SHRUB_CLASSES = [
    120, 121, 122,          # Shrubland, Evergreen shrubland, Deciduous shrubland
    152,                    # Sparse shrubland
    130,                    # Grassland
    150,                    # Sparse vegetation
    153                     # Sparse herbaceous cover
]

BARE_CLASSES  = [200,       # Bare areas
                 201,       # Consolidated bare areas
                 202        # Unconsolidated bare areas
]

WATER_CLASSES = [210]       # Water body

BUILT_CLASSES = [190]       # Impervious surface

WETLAND_CLASSES = [181,     # Swamp
                   182,     # Marsh
                   183,     # Flooded flat
                   184,     # Saline 
                   185,     # Mangrove
                   186,     # Salt Marsh
                   187      # Tidal flat
]

SNOW_CLASSES  = [220]

MAJOR_CLASSES = {
    1: CROP_CLASSES,
    2: TREE_CLASSES,
    3: SHRUB_CLASSES,
    4: BARE_CLASSES,
    5: WATER_CLASSES,
    6: BUILT_CLASSES,
    7: WETLAND_CLASSES,
    8: SNOW_CLASSES
}

def remap_to_major(image):
    from_vals = []
    to_vals = []

    for major_id, class_list in MAJOR_CLASSES.items():
        for c in class_list:
            from_vals.append(c)
            to_vals.append(major_id)

    return image.remap(from_vals, to_vals).rename("major_lc")


In [ ]:
aez = ee.FeatureCollection(
    "projects/ext-datasets/assets/datasets/Agro_Ecological_Zones"
)
india = aez.geometry().dissolve()

annual = ee.ImageCollection(
    "projects/sat-io/open-datasets/GLC-FCS30D/annual"
).mosaic()

start_year = 2000
end_year = 2022

for year in range(start_year, end_year + 1):
    
    band_name = f"b{year - 1999}" 
    
    glc = annual.select(band_name)
    
    major_lc = remap_to_major(glc).clip(india)

    task = ee.batch.Export.image.toDrive(
        image=major_lc,
        description=f"GLC_major_{year}",
        folder="GEE_GLC_India",
        fileNamePrefix=f"glc_major_{year}",
        region=india,
        scale=30,
        maxPixels=1e13
    )
    
    task.start()

In [ ]:
# merge 4 .tif files in QGIS